# SQL practice 11: DDL, constraints, views and modeling

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-11-ddl-constraints-and-views.ipynb)

12 exercises on table DDL, defaults, schema changes, constraints, data-quality checks, views and dimensional modeling. Each exercise names the [reference entries](sql-reference-11-ddl-constraints-and-views.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 delta-spark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 11](sql-reference-11-ddl-constraints-and-views.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell. This notebook uses a Delta Lake session (the Delta jars download on first use) so constraints are enforced. `fresh(table, select)` re-creates a Delta table and `drop(*tables)` removes tables.

In [ ]:
import os
import sys
import tempfile
import time

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="sql-")  # tables created in this notebook live here

builder = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Delta's SQL: MERGE, UPDATE, time travel
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # fetches the Delta jars on the first run
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

def fresh(table, select):
    """Drop and re-create a Delta table from a query."""
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    spark.sql(f"CREATE TABLE {table} USING DELTA AS {select}")


def drop(*tables):
    for table in tables:
        spark.sql(f"DROP TABLE IF EXISTS {table}")

## Exercises

7 easy, 4 medium, 1 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [A constrained table](#exercise-1-a-constrained-table) | Easy | [`CREATE TABLE`](sql-reference-11-ddl-constraints-and-views.ipynb#create-table), [`NOT NULL` and `CHECK` constraints](sql-reference-11-ddl-constraints-and-views.ipynb#not-null-and-check-constraints) |
| 2 | [Column defaults](#exercise-2-column-defaults) | Easy | [`CREATE TABLE`](sql-reference-11-ddl-constraints-and-views.ipynb#create-table) |
| 3 | [Add a column without breaking readers](#exercise-3-add-a-column-without-breaking-readers) | Easy | [`ALTER TABLE` and schema evolution](sql-reference-11-ddl-constraints-and-views.ipynb#alter-table-and-schema-evolution) |
| 4 | [Find duplicate keys](#exercise-4-find-duplicate-keys) | Easy | [Data quality checks in SQL](sql-reference-11-ddl-constraints-and-views.ipynb#data-quality-checks-in-sql) |
| 5 | [Orphaned foreign keys](#exercise-5-orphaned-foreign-keys) | Easy | [Data quality checks in SQL](sql-reference-11-ddl-constraints-and-views.ipynb#data-quality-checks-in-sql) |
| 6 | [A data quality report](#exercise-6-a-data-quality-report) | Medium | [Data quality checks in SQL](sql-reference-11-ddl-constraints-and-views.ipynb#data-quality-checks-in-sql) |
| 7 | [A view as a stable interface](#exercise-7-a-view-as-a-stable-interface) | Easy | [Temporary, global and persistent views](sql-reference-11-ddl-constraints-and-views.ipynb#temporary-global-and-persistent-views) |
| 8 | [Hash surrogate keys](#exercise-8-hash-surrogate-keys) | Easy | [Grain, facts and dimensions](sql-reference-11-ddl-constraints-and-views.ipynb#grain-facts-and-dimensions) |
| 9 | [Check a fact table's grain](#exercise-9-check-a-fact-tables-grain) | Medium | [Grain, facts and dimensions](sql-reference-11-ddl-constraints-and-views.ipynb#grain-facts-and-dimensions) |
| 10 | [Star schema query](#exercise-10-star-schema-query) | Medium | [Grain, facts and dimensions](sql-reference-11-ddl-constraints-and-views.ipynb#grain-facts-and-dimensions) |
| 11 | [Partition a table by day](#exercise-11-partition-a-table-by-day) | Medium | [`CREATE TABLE`](sql-reference-11-ddl-constraints-and-views.ipynb#create-table), [Inspecting tables](sql-reference-11-ddl-constraints-and-views.ipynb#inspecting-tables) |
| 12 | [A date dimension](#exercise-12-a-date-dimension) | Hard | [Grain, facts and dimensions](sql-reference-11-ddl-constraints-and-views.ipynb#grain-facts-and-dimensions) |

## Exercise 1: A constrained table

**Difficulty:** Easy · about 10 min · **Practises:** [`CREATE TABLE`](sql-reference-11-ddl-constraints-and-views.ipynb#create-table), [`NOT NULL` and `CHECK` constraints](sql-reference-11-ddl-constraints-and-views.ipynb#not-null-and-check-constraints)

*Encode the rules of a table in its definition.*

Create the Delta table `payments` with `payment_id INT NOT NULL`, `order_id INT NOT NULL`, `method STRING` and `amount DECIMAL(10,2)`, add a CHECK constraint `positive_amount` (amount > 0), insert the three payments listed in the input cell, then return all rows ordered by `payment_id`.

In [ ]:
def reset():
    drop("payments")
reset()
print("Insert: (1, 1, 'card', 120.00), (2, 4, 'card', 260.00), (3, 7, 'transfer', 310.00)")

**Expected output**

| payment_id | order_id | method | amount |
| --- | --- | --- | --- |
| 1 | 1 | card | 120.00 |
| 2 | 4 | card | 260.00 |
| 3 | 7 | transfer | 310.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    CREATE TABLE payments (
        payment_id INT NOT NULL,
        order_id   INT NOT NULL,
        method     STRING,
        amount     DECIMAL(10,2)
    ) USING DELTA
""")
spark.sql("""
    ALTER TABLE payments ADD CONSTRAINT positive_amount CHECK (amount > 0)
""")
spark.sql("""
    INSERT INTO payments VALUES (1, 1, 'card', 120.00), (2, 4, 'card', 260.00), (3, 7, 'transfer', 310.00)
""")
q = """
    SELECT payment_id, order_id, method, amount
    FROM payments
    ORDER BY payment_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Column defaults

**Difficulty:** Easy · about 5 min · **Practises:** [`CREATE TABLE`](sql-reference-11-ddl-constraints-and-views.ipynb#create-table)

*Defaults keep inserts short and consistent.*

Create the Delta table `tickets` (`ticket_id INT`, `priority STRING DEFAULT 'normal'`, `status STRING DEFAULT 'open'`) with column defaults enabled, insert tickets 1 and 2 giving only `ticket_id`, and ticket 3 with priority `high`, then return all rows ordered by `ticket_id`.

In [ ]:
def reset():
    drop("tickets")
reset()

**Expected output**

| ticket_id | priority | status |
| --- | --- | --- |
| 1 | normal | open |
| 2 | normal | open |
| 3 | high | open |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    CREATE TABLE tickets (
        ticket_id INT,
        priority  STRING DEFAULT 'normal',
        status    STRING DEFAULT 'open'
    ) USING DELTA
    TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
""")
spark.sql("""
    INSERT INTO tickets (ticket_id) VALUES (1), (2)
""")
spark.sql("""
    INSERT INTO tickets (ticket_id, priority) VALUES (3, 'high')
""")
q = """
    SELECT ticket_id, priority, status
    FROM tickets
    ORDER BY ticket_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: Add a column without breaking readers

**Difficulty:** Easy · about 5 min · **Practises:** [`ALTER TABLE` and schema evolution](sql-reference-11-ddl-constraints-and-views.ipynb#alter-table-and-schema-evolution)

*Additive schema changes are the safe kind.*

Add a nullable `channel STRING` column to `fact_orders`, set it to `web` for orders 1 to 3, then return `order_id`, `status` and `channel` for orders 1 to 5, ordered by `order_id`.

In [ ]:
def reset():
    fresh("fact_orders", "SELECT order_id, customer_id, status, to_date(order_ts) AS order_date, amount FROM orders")
reset()

**Expected output**

| order_id | status | channel |
| --- | --- | --- |
| 1 | paid | web |
| 2 | shipped | web |
| 3 | returned | web |
| 4 | paid | null |
| 5 | placed | null |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    ALTER TABLE fact_orders ADD COLUMNS (channel STRING)
""")
spark.sql("""
    UPDATE fact_orders SET channel = 'web' WHERE order_id <= 3
""")
q = """
    SELECT order_id, status, channel
    FROM fact_orders
    WHERE order_id <= 5
    ORDER BY order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: Find duplicate keys

**Difficulty:** Easy · about 5 min · **Practises:** [Data quality checks in SQL](sql-reference-11-ddl-constraints-and-views.ipynb#data-quality-checks-in-sql)

*Lakehouse tables do not enforce primary keys, so pipelines check them.*

Using `landed_customers` below, return each `customer_id` that appears more than once with `copies` and `distinct_names`, ordered by `customer_id`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW landed_customers AS SELECT * FROM VALUES
  (1, 'Acme Corp'), (2, 'Bright Retail'), (2, 'Bright Retail'), (3, 'Cyan Traders'), (3, 'Cyan Trading'), (3, 'Cyan Traders')
AS c(customer_id, name)
""")

**Expected output**

| customer_id | copies | distinct_names |
| --- | --- | --- |
| 2 | 2 | 1 |
| 3 | 3 | 2 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, count(*) AS copies, count(DISTINCT name) AS distinct_names
    FROM landed_customers
    GROUP BY customer_id
    HAVING count(*) > 1
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: Orphaned foreign keys

**Difficulty:** Easy · about 5 min · **Practises:** [Data quality checks in SQL](sql-reference-11-ddl-constraints-and-views.ipynb#data-quality-checks-in-sql)

*Facts that point to missing dimension rows break joins silently.*

Using `landed_orders` below, return `order_id` and `customer_id` of orders whose customer is not in `customers`, including orders with a NULL customer, ordered by `order_id`.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW landed_orders AS SELECT * FROM VALUES (101, 1), (102, 42), (103, NULL), (104, 8) AS o(order_id, customer_id)")

**Expected output**

| order_id | customer_id |
| --- | --- |
| 102 | 42 |
| 103 | null |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT o.order_id, o.customer_id
    FROM landed_orders o
    LEFT ANTI JOIN customers c ON c.customer_id = o.customer_id
    ORDER BY o.order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: A data quality report

**Difficulty:** Medium · about 15 min · **Practises:** [Data quality checks in SQL](sql-reference-11-ddl-constraints-and-views.ipynb#data-quality-checks-in-sql)

*One query that every load runs before publishing.*

Return `rule` and `violations` for four rules over the shared tables, in this order: `order id unique`, `order customer exists`, `item sku in catalogue`, `amount positive`.

**Expected output**

| rule | violations |
| --- | --- |
| order id unique | 0 |
| order customer exists | 0 |
| item sku in catalogue | 1 |
| amount positive | 0 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT 'order id unique' AS rule, count(*) AS violations
    FROM (SELECT order_id FROM orders GROUP BY order_id HAVING count(*) > 1)
    UNION ALL
    SELECT 'order customer exists', count(*)
    FROM orders o LEFT ANTI JOIN customers c ON c.customer_id = o.customer_id
    UNION ALL
    SELECT 'item sku in catalogue', count(*)
    FROM order_items i LEFT ANTI JOIN products p ON p.sku = i.sku
    UNION ALL
    SELECT 'amount positive', count_if(amount <= 0)
    FROM orders
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: A view as a stable interface

**Difficulty:** Easy · about 5 min · **Practises:** [Temporary, global and persistent views](sql-reference-11-ddl-constraints-and-views.ipynb#temporary-global-and-persistent-views)

*Consumers read a view, so the tables behind it can change.*

Create (or replace) the temporary view `v_active_customers` with `customer_id`, `name` and `orders` for customers with at least one non-returned order, then return its rows ordered by `orders` descending, `customer_id`. (A persistent view cannot reference the temporary sample views.)

**Expected output**

| customer_id | name | orders |
| --- | --- | --- |
| 1 | Acme Corp | 2 |
| 2 | Bright Retail | 2 |
| 3 | Cyan Traders | 2 |
| 6 | Foxtrot Foods | 2 |
| 4 | Delta Grocers | 1 |
| 5 | Echo Supplies | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW v_active_customers AS
    SELECT c.customer_id, c.name, count(*) AS orders
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    WHERE o.status <> 'returned'
    GROUP BY c.customer_id, c.name
""")
q = """
    SELECT customer_id, name, orders
    FROM v_active_customers
    ORDER BY orders DESC, customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Hash surrogate keys

**Difficulty:** Easy · about 5 min · **Practises:** [Grain, facts and dimensions](sql-reference-11-ddl-constraints-and-views.ipynb#grain-facts-and-dimensions)

*Deterministic keys that are identical in every environment.*

Return `sku`, `category` and `product_key` (`xxhash64` of the SKU) for Garden products, ordered by `sku`.

**Expected output**

| sku | category | product_key |
| --- | --- | --- |
| SKU-4003 | Garden | 559238310115464606 |
| SKU-4004 | Garden | 3100379872642230831 |
| SKU-4010 | Garden | 8927697464848921068 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT sku, category, xxhash64(sku) AS product_key
    FROM products
    WHERE category = 'Garden'
    ORDER BY sku
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: Check a fact table's grain

**Difficulty:** Medium · about 10 min · **Practises:** [Grain, facts and dimensions](sql-reference-11-ddl-constraints-and-views.ipynb#grain-facts-and-dimensions)

*Every fact build should prove its grain.*

Build `fact_order_line` (one row per order line: `order_id`, `sku`, `order_date`, `customer_id`, `qty`, `line_amount`) with CTAS, then return one row with `rows`, `grain_keys` (distinct order_id and sku pairs) and `grain_ok` (whether they are equal).

**Expected output**

| rows | grain_keys | grain_ok |
| --- | --- | --- |
| 17 | 17 | true |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE fact_order_line USING DELTA AS
    SELECT i.order_id, i.sku, to_date(o.order_ts) AS order_date, o.customer_id, i.qty, i.qty * i.unit_price AS line_amount
    FROM order_items i
    JOIN orders o ON o.order_id = i.order_id
""")
q = """
    SELECT count(*) AS rows,
           count(DISTINCT order_id, sku) AS grain_keys,
           count(*) = count(DISTINCT order_id, sku) AS grain_ok
    FROM fact_order_line
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: Star schema query

**Difficulty:** Medium · about 10 min · **Practises:** [Grain, facts and dimensions](sql-reference-11-ddl-constraints-and-views.ipynb#grain-facts-and-dimensions)

*Facts joined to two dimensions, the shape of most BI queries.*

Return `region` (`Unknown` when missing), `category` (`Unknown` when missing) and `revenue` (sum of `qty * unit_price`) from `order_items`, `orders`, `customers` and `products`, ordered by `revenue` descending, then `region`, `category`.

**Expected output**

| region | category | revenue |
| --- | --- | --- |
| Unknown | Pantry | 1180.00 |
| South | Outdoor | 310.00 |
| North | Garden | 260.00 |
| West | Outdoor | 210.00 |
| North | Kitchen | 184.50 |
| South | Garden | 149.00 |
| North | Pantry | 105.25 |
| Unknown | Garden | 89.90 |
| West | Kitchen | 74.00 |
| West | Pantry | 55.00 |
| West | Unknown | 41.40 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT coalesce(c.region, 'Unknown') AS region,
           coalesce(p.category, 'Unknown') AS category,
           sum(i.qty * i.unit_price) AS revenue
    FROM order_items i
    JOIN orders o ON o.order_id = i.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    LEFT JOIN products p ON p.sku = i.sku
    GROUP BY coalesce(c.region, 'Unknown'), coalesce(p.category, 'Unknown')
    ORDER BY revenue DESC, region, category
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Partition a table by day

**Difficulty:** Medium · about 10 min · **Practises:** [`CREATE TABLE`](sql-reference-11-ddl-constraints-and-views.ipynb#create-table), [Inspecting tables](sql-reference-11-ddl-constraints-and-views.ipynb#inspecting-tables)

*Partitioning by the most common filter column lets queries skip data.*

Create (or replace) the Delta table `orders_by_day` partitioned by `order_date` with `order_id`, `customer_id`, `amount` and `order_date` from `orders`, then return `order_date` and `orders` for 2026-01-12 onwards, ordered by `order_date`.

**Expected output**

| order_date | orders |
| --- | --- |
| 2026-01-12 | 2 |
| 2026-01-13 | 1 |
| 2026-01-14 | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE orders_by_day
    USING DELTA
    PARTITIONED BY (order_date)
    AS SELECT order_id, customer_id, amount, to_date(order_ts) AS order_date FROM orders
""")
q = """
    SELECT order_date, count(*) AS orders
    FROM orders_by_day
    WHERE order_date >= DATE'2026-01-12'
    GROUP BY order_date
    ORDER BY order_date
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: A date dimension

**Difficulty:** Hard · about 15 min · **Practises:** [Grain, facts and dimensions](sql-reference-11-ddl-constraints-and-views.ipynb#grain-facts-and-dimensions)

*Every warehouse needs a calendar table.*

Create (or replace) the Delta table `dim_date` with one row per day from 2026-01-01 to 2026-03-31: `date_key` (INT `yyyyMMdd`), `day`, `weekday` (short name), `is_weekend`, `month` (e.g. `2026-01`) and `week_start` (the Monday). Return the rows from 2026-01-30 to 2026-02-02, ordered by `day`.

**Expected output**

| date_key | day | weekday | is_weekend | month | week_start |
| --- | --- | --- | --- | --- | --- |
| 20260130 | 2026-01-30 | Fri | false | 2026-01 | 2026-01-26 |
| 20260131 | 2026-01-31 | Sat | true | 2026-01 | 2026-01-26 |
| 20260201 | 2026-02-01 | Sun | true | 2026-02 | 2026-01-26 |
| 20260202 | 2026-02-02 | Mon | false | 2026-02 | 2026-02-02 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE dim_date USING DELTA AS
    SELECT CAST(date_format(day, 'yyyyMMdd') AS INT) AS date_key,
           day,
           date_format(day, 'EEE') AS weekday,
           dayofweek(day) IN (1, 7) AS is_weekend,
           date_format(day, 'yyyy-MM') AS month,
           to_date(date_trunc('week', day)) AS week_start
    FROM (SELECT explode(sequence(DATE'2026-01-01', DATE'2026-03-31', INTERVAL 1 DAY)) AS day)
""")
q = """
    SELECT date_key, day, weekday, is_weekend, month, week_start
    FROM dim_date
    WHERE day BETWEEN DATE'2026-01-30' AND DATE'2026-02-02'
    ORDER BY day
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 11: DDL, constraints, views and modeling](sql-reference-11-ddl-constraints-and-views.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).